# Lógica

La **lógica proposicional** representa el conocimiento como oraciones que son verdaderas o falsas, construidas a partir de proposiciones simples con los conectores *no* (¬), *y* (∧), *o* (∨), *implica* (→) y *si y solo si* (↔). Una **base de conocimiento** es un conjunto de oraciones que se toman como verdaderas. **Implica** (*entails*) una oración α, que se escribe KB ⊨ α, cuando α es verdadera en **todos** los mundos en los que la base de conocimiento es verdadera.

## Oraciones como objetos de Python

Cada tipo de oración es una pequeña clase que puede evaluarse a sí misma en un **modelo**, un diccionario que da un valor de verdad a cada proposición, y que puede listar las proposiciones que usa. Los operadores `~`, `&` y `|` construyen las oraciones.

In [ ]:
class Sentence:
    def __invert__(self):
        return Not(self)

    def __and__(self, other):
        return And(self, other)

    def __or__(self, other):
        return Or(self, other)


class Symbol(Sentence):
    def __init__(self, name):
        self.name = name

    def evaluate(self, model):
        return model[self.name]

    def symbols(self):
        return {self.name}

    def __repr__(self):
        return self.name


class Not(Sentence):
    def __init__(self, operand):
        self.operand = operand

    def evaluate(self, model):
        return not self.operand.evaluate(model)

    def symbols(self):
        return self.operand.symbols()

    def __repr__(self):
        return f"¬{self.operand}"


class Connective(Sentence):
    symbol = "?"

    def __init__(self, left, right):
        self.left, self.right = left, right

    def symbols(self):
        return self.left.symbols() | self.right.symbols()

    def __repr__(self):
        return f"({self.left} {self.symbol} {self.right})"


class And(Connective):
    symbol = "∧"

    def evaluate(self, model):
        return self.left.evaluate(model) and self.right.evaluate(model)


class Or(Connective):
    symbol = "∨"

    def evaluate(self, model):
        return self.left.evaluate(model) or self.right.evaluate(model)


class Implies(Connective):
    symbol = "→"

    def evaluate(self, model):
        return not self.left.evaluate(model) or self.right.evaluate(model)


class Iff(Connective):
    symbol = "↔"

    def evaluate(self, model):
        return self.left.evaluate(model) == self.right.evaluate(model)

## Implicación mediante comprobación de modelos

La forma más directa de decidir la implicación es la **comprobación de modelos** (*model checking*): listar todos los mundos posibles y comprobar que la consulta es verdadera en cada uno en el que lo es la base de conocimiento. Con *n* proposiciones hay 2ⁿ mundos, así que solo sirve para problemas pequeños; los razonadores reales usan reglas de inferencia, como la **resolución**, o resolvedores SAT.

Una pregunta tiene entonces tres respuestas posibles, no dos: *sí* si la base de conocimiento la implica, *no* si implica su negación, y *desconocido* si no implica ninguna de las dos.

In [ ]:
from itertools import product


def entails(knowledge_base, query):
    names = sorted(set().union(*(sentence.symbols() for sentence in knowledge_base + [query])))
    for values in product([False, True], repeat=len(names)):
        model = dict(zip(names, values))
        if all(sentence.evaluate(model) for sentence in knowledge_base) and not query.evaluate(model):
            return False
    return True


def answer(knowledge_base, query):
    if entails(knowledge_base, query):
        return "yes"
    if entails(knowledge_base, ~query):
        return "no"
    return "unknown"

## Un caso de detectives

Exactamente una de estas personas, Alice, Bob y Carol, es culpable. Alice solo lo habría hecho con la ayuda de Carol, y Carol tiene coartada.

In [ ]:
alice, bob, carol = Symbol("alice"), Symbol("bob"), Symbol("carol")

exactly_one_guilty = [alice | bob | carol, ~(alice & bob), ~(alice & carol), ~(bob & carol)]
clues = [Implies(alice, carol), ~carol]

for suspect in [alice, bob, carol]:
    print(f"Is {suspect} guilty? {answer(exactly_one_guilty + clues, suspect)}")

Is alice guilty? no
Is bob guilty? yes
Is carol guilty? no


Sin la coartada de Carol, la base de conocimiento sigue descartando a Alice, pero no puede decidir entre Bob y Carol:

In [ ]:
for suspect in [alice, bob, carol]:
    print(f"Is {suspect} guilty? {answer(exactly_one_guilty + [Implies(alice, carol)], suspect)}")

Is alice guilty? no
Is bob guilty? unknown
Is carol guilty? unknown


*Desconocido* es una respuesta honesta que una función de sí o no ocultaría: un programa que respondiera *no* aquí estaría afirmando la inocencia de Bob sin fundamento.

## Programación lógica

La lógica proposicional trata con afirmaciones fijas: *alice es culpable*. La **lógica de primer orden** añade objetos, relaciones entre ellos y variables: *para todo X, Y y Z, si X es progenitor de Y e Y es antepasado de Z, entonces X es antepasado de Z*. Una sola oración cubre entonces todas las familias.

La **programación lógica** ejecuta oraciones así como programas: declaras hechos y reglas, haces una pregunta, y el lenguaje busca las respuestas. Su lenguaje principal es **Prolog** (Alain Colmerauer y Philippe Roussel, 1972). Lisp (John McCarthy, 1958), el otro lenguaje clásico de la IA, no es un lenguaje lógico: es funcional, y la IA simbólica lo usaba para escribir programas como los sistemas expertos.

Una familia en Prolog. Un nombre en minúsculas es una constante, uno con mayúscula inicial una variable, `:-` se lee *si* y la coma *y*:

```prolog
parent(ana, bea).
parent(bea, carla).
parent(carla, dani).
parent(ana, eva).

ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).
```

Tres preguntas, con las respuestas de SWI-Prolog, la implementación libre más usada:

```prolog
?- findall(Who, ancestor(ana, Who), Descendants).
Descendants = [bea, eva, carla, dani].

?- findall(Who, ancestor(Who, dani), Ancestors).
Ancestors = [carla, ana, bea].

?- ancestor(eva, dani).
false.
```

Puedes ejecutarlas, sin instalar nada, en [SWISH](https://swish.swi-prolog.org), el entorno en línea de SWI-Prolog.

Las mismas dos reglas responden preguntas opuestas, quién desciende de Ana y de quién desciende Dani, porque un programa de Prolog declara relaciones, no procedimientos. Prolog encuentra las respuestas por **encadenamiento hacia atrás**, la estrategia de MYCIN en [el sistema experto](rules_and_expert_systems.ipynb): prueba las cláusulas en el orden en que están escritas, en profundidad, y por eso las respuestas salen en ese orden.

`false` significa *no demostrable a partir de estos hechos*. Prolog supone que todo lo que no puede demostrar es falso, la **hipótesis del mundo cerrado**, donde el comprobador de modelos de arriba habría respondido *desconocido*: un hecho ausente cuenta como falso, igual que una fila ausente en una base de datos.

Prolog es hoy un lenguaje de nicho, que se usa donde las reglas son la descripción natural de un problema: Watson de IBM, por ejemplo, analizaba las preguntas del concurso *Jeopardy!* con reglas de Prolog cuando ganó en 2011. Su pariente más sencillo, **Datalog**, encontró una segunda vida en herramientas que analizan código fuente: CodeQL de GitHub consulta una base de código en un dialecto de Datalog.

## Reglas recursivas en SQL

La idea que llegó a todos los desarrolladores es la recursión sobre relaciones. Desde el estándar SQL:1999, las bases de datos SQL responden **consultas recursivas**, que calculan la misma relación de *antepasado* a partir de una tabla de progenitores. SQLite, que viene con Python, ejecuta una:

In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE parent (older TEXT, younger TEXT)")
connection.executemany("INSERT INTO parent VALUES (?, ?)",
                       [("ana", "bea"), ("bea", "carla"), ("carla", "dani"), ("ana", "eva")])

In [ ]:
ancestors_query = """
WITH RECURSIVE ancestor(older, younger) AS (
    SELECT older, younger FROM parent
    UNION
    SELECT parent.older, ancestor.younger
    FROM parent JOIN ancestor ON parent.younger = ancestor.older
)
SELECT older, younger FROM ancestor ORDER BY older, younger
"""

for older, younger in connection.execute(ancestors_query):
    print(f"{older} is an ancestor of {younger}")

ana is an ancestor of bea
ana is an ancestor of carla
ana is an ancestor of dani
ana is an ancestor of eva
bea is an ancestor of carla
bea is an ancestor of dani
carla is an ancestor of dani


Las dos partes de la consulta son las dos reglas de Prolog: el primer `SELECT` dice que un progenitor es un antepasado, y el segundo que un progenitor de un antepasado es un antepasado. La base de datos la evalúa por **encadenamiento hacia delante**: cada pasada une los progenitores con los pares encontrados hasta el momento, hasta que una pasada no añade nada nuevo, el punto fijo de [el sistema experto](rules_and_expert_systems.ipynb). Las consultas recursivas son la forma en que las bases de datos recorren organigramas, árboles de carpetas y las listas de piezas de los productos fabricados.